In [50]:
import pandas as pd
import numpy as np
import sklearn
import matplotlib as plt

## Preprocessing workflow

This notebook prepares a regression-ready dataset of U.S. Craigslist listings for the research question. It uses `pandas` and `numpy` for data preparation; modeling and visualization packages are also imported.

In [51]:
df = pd.read_csv('vehicles.csv')

The dataset is loaded into `df`.

In [52]:
print(df.shape)
df.columns.tolist()
df.head()
df.dtypes

(426880, 26)


id                int64
url              object
region           object
region_url       object
price             int64
year            float64
manufacturer     object
model            object
condition        object
cylinders        object
fuel             object
odometer        float64
title_status     object
transmission     object
VIN              object
drive            object
size             object
type             object
paint_color      object
image_url        object
description      object
county          float64
state            object
lat             float64
long            float64
posting_date     object
dtype: object

Inspection the dataset dimensions, column names, and data types before filtering. This check identifies the exact variables needed for the study and helps detect unexpected data formats early.

In [53]:
print("Duplicate URLs:", df["url"].duplicated().sum())
print("Duplicate VINs:", df["VIN"].duplicated().sum())
print("Missing VINs:", df["VIN"].isna().sum())

Duplicate URLs: 0
Duplicate VINs: 308615
Missing VINs: 161042


No duplicate listing URLs were found, so no URL-based deduplication is needed. VIN cannot be used as a unique listing identifier because it has a lot of missing values and identifies a physical vehicle, while our research focus is on individual Craigslist listings. The same vehicle may be in multiple listings with different prices or mileages. Removing those records could result in loss useful information.

In [54]:
print(
    df["posting_date"]
    .astype("string")
    .str[:7]
    .value_counts()
    .sort_index()
)

posting_date
2021-04    316969
2021-05    109843
Name: count, dtype: Int64


Our intended time frame is April to May 2021 and this confirms that the dataset has listings from that date: 316,969 records were posted in April 2021 and 109,843 in May 2021.

In [55]:
print(df["posting_date"].isna().sum())

68


This cell checks whether any records lack a posting date. Missing dates make it hard to know if the listing is withing our intended timeframe.

In [56]:
print(
    df[df["type"].isin(["pickup", "sedan"])]["posting_date"].isna().sum()
)

0


There were 68 missing vdate values, and none of them are in pickups or sedans which is our interest.

In [57]:
df = df.dropna(subset=["posting_date"])

Records with missing `posting_date` are removed because we are not sure if they were made between April and May 2021. No other date filter is needed because the observed non-missing dates are already April or May 2021.

In [58]:
print(df["posting_date"].isna().sum())

0


The follow-up check confirms that the working dataset has no missing posting dates after the exclusion.

In [59]:
subset_columns = df[["id","price", "odometer", "year", "manufacturer", "condition", "transmission", "drive", "type"]]

Create a subset with the variables required by the research question: listing price, odometer reading, model year, manufacturer, condition, transmission, drivetrain, and vehicle type. Limiting the working table to these variables makes cleaning decisions easier.

In [60]:
subset_type = subset_columns[
    subset_columns["type"].isin(["pickup", "sedan"])
]

We restrict the study to listings whose `type` is exactly `pickup` or `sedan`. The dataset contains a separate `truck` category, which is not combined with `pickup` because in oiur research, we are interested in pickups and sedans.

In [61]:
print("Duplicate IDs:", subset_type["id"].duplicated().sum())

Duplicate IDs: 0


There are no duplicate listing IDs in the pickup/sedan subset. We therefore retain every filtered listing and do not remove rows merely because their non-identifier fields happen to match.

In [62]:
print(subset_type["type"].value_counts(dropna=False))
print(subset_type.isna().sum())
print(subset_type[["price", "odometer", "year"]].describe())

type
sedan     87056
pickup    43510
Name: count, dtype: int64
id                  0
price               0
odometer         1542
year              578
manufacturer     3011
condition       44092
transmission      599
drive           28275
type                0
dtype: int64
              price      odometer           year
count  1.305660e+05  1.290240e+05  129988.000000
mean   5.887217e+04  8.804216e+04    2012.435240
std    1.081098e+07  1.585760e+05       7.830641
min    0.000000e+00  0.000000e+00    1923.000000
25%    6.350000e+03  3.433750e+04    2010.000000
50%    1.482800e+04  7.761300e+04    2014.000000
75%    2.699800e+04  1.245320e+05    2017.000000
max    3.736929e+09  1.000000e+07    2022.000000


This reports group sizes, missing values, and the distributions of price, mileage, and model year. It identifies missing characteristics and shows that the numeric variables contain zero and extremely large values need to evaluated.

In [63]:
numeric_cols = ["price", "odometer", "year"]

subset_type[numeric_cols].quantile([0, 0.01, 0.05, 0.50, 0.95, 0.99, 0.995, 0.999, 1])

for column in numeric_cols:
    print(f"\n{column}")
    print(subset_type.nlargest(10, column)[["type", "price", "odometer", "year"]])


price
          type       price   odometer    year
318592  pickup  3736928711   164000.0  2007.0
29386   pickup  1111111111   149000.0  1999.0
137807   sedan   123456789    96000.0  1999.0
193736   sedan   123456789    64181.0  2015.0
207080  pickup   123456789  1234567.0  2003.0
307488  pickup   123456789   320000.0  1996.0
136516  pickup    17000000   170000.0  2007.0
327938   sedan     3226714     1234.0  2002.0
68935    sedan     2000000   164290.0  2002.0
208617  pickup      554900    25365.0  2019.0

odometer
          type  price    odometer    year
144436   sedan   4500  10000000.0  2010.0
151740   sedan    800  10000000.0  1949.0
262338  pickup   4800  10000000.0  1949.0
287115   sedan      0  10000000.0  1995.0
292399   sedan   1300  10000000.0  1950.0
305172   sedan   4500  10000000.0  1995.0
27200    sedan   1500   9999999.0  1989.0
217319  pickup   6500   9999999.0  1941.0
217320  pickup   5500   9999999.0  1939.0
219681   sedan    900   9999999.0  1993.0

year
         

Percentiles and large observations are observed before setting thresholds. The data includes unrealistic values, such as prices in the hundreds of millions or billions and odometer readings near 10 million miles, so they should not be treated as vehicle listings.

In [64]:
rules = {
    "nonpositive price": subset_type["price"] <= 0,
    "nonpositive odometer": subset_type["odometer"] <= 0,
    "year after 2021": subset_type["year"] > 2021,
    "year before 1990": subset_type["year"] < 1990,
    "price above 100,000": subset_type["price"] > 100_000,
    "odometer above 300,000": subset_type["odometer"] > 300_000,
}

pd.Series({name: rule.sum() for name, rule in rules.items()})

nonpositive price         8871
nonpositive odometer       397
year after 2021              2
year before 1990          1920
price above 100,000         60
odometer above 300,000     418
dtype: int64

This documents the number of records with negative price or mileage, future model years, older model years, and high price or mileage values. The `year < 1990`, we keep older vehicles because they may be real vehicles.

In [65]:
subset_type.duplicated().sum()

np.int64(0)

In [66]:
subset_type.groupby("type")[["price", "odometer", "year"]].quantile(
    [0.01, 0.05, 0.50, 0.95, 0.99, 0.995, 0.999]
)

price    odometer      year
type                                         
pickup 0.010      0.000     102.000  1973.000
       0.050      1.000    7663.050  2000.000
       0.500  27990.000   71288.000  2015.000
       0.950  54466.000  211000.000  2020.000
       0.990  71995.000  274820.000  2020.000
       0.995  78981.000  307085.785  2020.365
       0.999  89900.000  999999.000  2021.000
sedan  0.010      0.000     160.000  1986.000
       0.050      0.000    8973.000  2002.000
       0.500   9995.000   79983.000  2014.000
       0.950  34947.000  184443.750  2019.000
       0.990  44000.000  234205.950  2020.000
       0.995  49995.000  260000.000  2020.000
       0.999  80942.165  400000.000  2021.000

We find quantiles for both pickups and sedans because the two vehicle types can have different price and mileage distributions. The results show that a price ceiling of $100,000 is above the 99.9th percentile for both groups.

In [67]:
subset_type.loc[
    (subset_type["price"] > 100_000) | (subset_type["odometer"] > 300_000),
    ["type", "price", "odometer", "year"]
].sort_values(["price", "odometer"], ascending=False).head(30)

,type,price,odometer,year
318592,pickup,3736928711,164000.0,2007.0
29386,pickup,1111111111,149000.0,1999.0
207080,pickup,123456789,1234567.0,2003.0
307488,pickup,123456789,320000.0,1996.0
137807,sedan,123456789,96000.0,1999.0
193736,sedan,123456789,64181.0,2015.0
136516,pickup,17000000,170000.0,2007.0
327938,sedan,3226714,1234.0,2002.0
68935,sedan,2000000,164290.0,2002.0
208617,pickup,554900,25365.0,2019.0


In [68]:
cleaned = subset_type.copy()

cleaned = cleaned.loc[
    cleaned["price"].between(101, 100_000)
    & cleaned["odometer"].between(1, 400_000)
    & cleaned["year"].between(1900, 2021)
].copy()

categorical_cols = ["manufacturer", "condition", "transmission", "drive"]
cleaned[categorical_cols] = cleaned[categorical_cols].fillna("unknown")

cleaned["vehicle_age"] = 2021 - cleaned["year"]
cleaned["odometer_10k"] = cleaned["odometer"] / 10_000

### Final cleaning decisions

The analysis keeps records with price from $101 to $100,000, odometer readings from 1 to 400,000 miles, and model years from 1900 through 2021. Prices of $100 or below are removed because processing shows unrealistic prices such as $1 and $7.

Rows missing `price`, `odometer`, or `year` are excluded by these numeric filters because those variables are required for the study, and age calculation. We keep missing `manufacturer`, `condition`, `transmission`, and `drive` as `unknown` for now and will fill them later with `.fillna()` or `.interpolate()`.

`vehicle_age` is calculated as `2021 - year`. `odometer_10k` equals odometer miles divided by 10,000, so a one-unit change represents the 10,000.

In [69]:
print(cleaned.shape)
print(cleaned.isna().sum())
print(cleaned["type"].value_counts())
print(cleaned[["price", "odometer", "vehicle_age", "odometer_10k"]].describe())

cleaned.groupby("type")[["price", "odometer", "vehicle_age", "odometer_10k"]].describe()

(119016, 11)
id              0
price           0
odometer        0
year            0
manufacturer    0
condition       0
transmission    0
drive           0
type            0
vehicle_age     0
odometer_10k    0
dtype: int64
type
sedan     78535
pickup    40481
Name: count, dtype: int64
               price       odometer    vehicle_age   odometer_10k
count  119016.000000  119016.000000  119016.000000  119016.000000
mean    19059.832502   85900.689277       8.667793       8.590069
std     13796.877490   60119.054091       7.876928       6.011905
min       101.000000       1.000000       0.000000       0.000100
25%      7800.000000   34375.000000       4.000000       3.437500
50%     15995.000000   78781.000000       7.000000       7.878100
75%     27990.000000  125957.000000      11.000000      12.595700
max     99999.000000  400000.000000      98.000000      40.000000


price                                                                \
          count          mean           std    min      25%      50%      75%   
type                                                                            
pickup  40481.0  28496.301598  14369.891345  123.0  18995.0  27995.0  35995.0   
sedan   78535.0  14195.788375  10605.553952  101.0   6000.0  10999.0  19990.0   

                odometer                ... vehicle_age       odometer_10k  \
            max    count          mean  ...         75%   max        count   
type                                    ...                                  
pickup  99999.0  40481.0  85517.186384  ...        10.0  98.0      40481.0   
sedan   99950.0  78535.0  86098.366499  ...        12.0  97.0      78535.0   

                                                                 
            mean       std     min     25%     50%    75%   max  
type                                                             
pickup  8.551719  6.729232  0.0001  2.7715  7.0559  13.20  40.0  
sedan   8.609837  5.606369  0.0001  3.8867  8.1675  12.37  40.0  

[2 rows x 32 columns]

The validation confirms the final dataset contains 119,781 complete records across the ten analysis variables: 40,544 pickups and 79,237 sedans.

In [70]:
lower_tail = {
    "price <= $100": (cleaned["price"] <= 100).sum(),
    "price <= $500": (cleaned["price"] <= 500).sum(),
    "odometer <= 100": (cleaned["odometer"] <= 100).sum(),
    "odometer <= 500": (cleaned["odometer"] <= 500).sum(),
}

pd.Series(lower_tail)

cleaned.loc[
    (cleaned["price"] <= 100) | (cleaned["odometer"] <= 100),
    ["type", "price", "odometer", "year", "manufacturer", "condition"]
].head(30)

,type,price,odometer,year,manufacturer,condition
2456,pickup,38615,1.0,2021.0,chevrolet,unknown
3501,pickup,38615,1.0,2021.0,chevrolet,unknown
6060,sedan,11992,1.0,2015.0,dodge,unknown
6065,sedan,34999,12.0,2019.0,mini,unknown
6331,sedan,11992,1.0,2015.0,dodge,unknown
6659,sedan,34499,53.0,2019.0,mini,unknown
6693,sedan,11992,1.0,2015.0,dodge,unknown
7042,sedan,34999,12.0,2019.0,mini,unknown
7107,sedan,11992,1.0,2015.0,dodge,unknown
7467,sedan,11992,1.0,2015.0,dodge,unknown


The zero count for prices at or below $100 is expected because the final price rule has already removed them. The displayed low-mileage listings include new 2021 vehicles and older vehicles; because these can be real, no lower odometer cutoff.

In [71]:
print(pd.Series(lower_tail))

price <= $100         0
price <= $500      1786
odometer <= 100     606
odometer <= 500    1098
dtype: int64


In [73]:
cleaned.to_csv("craigslist_pickup_sedan_cleaned.csv", index=False)

The completed analytic dataset is saved as `craigslist_pickup_sedan_cleaned.csv` without the pandas index. This file is the reproducible input for the next checkpoint's exploratory plots and adjusted price model.